# Word count CORD-19 - Giulia

Questo notebook lavora in locale sul Mac, dentro la cartella del progetto.

Input atteso:

```text
data/silver/paragraphs
```

Questa cartella deve essere copiata dalla VM con `rsync`. Non e' un singolo file: e' una directory Parquet con piu partizioni.

Il notebook legge solo il dataset silver gia sanificato. Non rigenera `silver/paragraphs` e non tocca i dati originali.

## Nota sul memory leak

Il problema descritto nel report del memory leak riguardava la fase di creazione di `silver/paragraphs`, in particolare un uso costoso di `Series.isin(...)` con un grande set Python dentro operazioni per partizione.

Questo notebook evita quella zona fragile: parte dal dataset `silver/paragraphs` gia pronto e lo legge in formato Parquet. Per ridurre il rischio di pressione sulla memoria:

- legge solo le colonne necessarie;
- usa Dask in modo lazy;
- fa prima smoke test piccoli;
- salva output locali leggeri in CSV/JSON;
- non usa grandi set/list/dict esterni dentro `map_partitions`.


## 1. Setup dei path e dei parametri

Questa cella definisce i percorsi locali del progetto.

Gli output vengono salvati in una cartella personale sotto `Giulia/reports/giulia_word_count`, cosi non si mescolano con eventuali output di altri task.

In [ ]:
from pathlib import Path
import json
import os
import time

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "DATA_DICTIONARY.md").exists():
            return candidate
    raise RuntimeError("Non trovo la root del progetto: apri la cartella MAPD-Project-main in VS Code.")

ROOT = find_project_root(Path.cwd())
DATA = ROOT / "data"
INPUT = DATA / "silver" / "paragraphs"
OUT = ROOT / "Giulia" / "reports" / "giulia_word_count"
OUT.mkdir(parents=True, exist_ok=True)

DOC_COL = "cord_uid"
TEXT_COL = "text"
REFERENCE_COL = "is_reference_like"

TOP_N = 50
SPLIT_OUT = 4

os.environ["CORD19_DATA"] = str(DATA)

print("project root:", ROOT)
print("input       :", INPUT)
print("output      :", OUT)

## 2. Controllo ambiente e schema Parquet

Questa cella verifica che le librerie siano disponibili e apre il dataset Parquet con PyArrow.

La stampa dello schema legge i metadati, non carica tutto il corpus in memoria.

In [ ]:
import dask
import dask.dataframe as dd
import pandas as pd
import pyarrow.dataset as ds

if not INPUT.exists():
    raise FileNotFoundError(
        f"Dataset non trovato: {INPUT}\n"
        "Aspetta che rsync finisca oppure controlla di essere nella cartella locale del progetto."
    )

dataset = ds.dataset(INPUT, format="parquet")
print(dataset.schema)

## 3. Avvio controllato di Dask

Questa cella avvia un cluster Dask locale sul Mac.

La configurazione e' volutamente prudente: un solo worker, un thread per worker e un limite di memoria piu ampio. Sul Mac e' piu lenta di una configurazione parallela, ma riduce il rischio di riavvii del worker per pressione di memoria.


In [ ]:
from dask.distributed import Client, LocalCluster

if "client" in globals():
    try:
        client.close()
    except Exception:
        pass

if "cluster" in globals():
    try:
        cluster.close()
    except Exception:
        pass

cluster = LocalCluster(
    n_workers=1,
    threads_per_worker=1,
    memory_limit="6GB",
    processes=True,
    dashboard_address=":0",
)

client = Client(cluster)
client

## 4. Lettura lazy del dataset silver

Questa cella legge `silver/paragraphs` con Dask DataFrame.

Vengono lette solo le colonne necessarie al word count:

- `cord_uid`, identificativo del paper;
- `text`, testo del paragrafo;
- `is_reference_like`, flag per sezioni tipo references/acknowledgements/funding.

La chiamata `head(3)` carica solo poche righe per controllare che tutto sia leggibile.

In [ ]:
columns = set(dataset.schema.names)
required = {DOC_COL, TEXT_COL}
missing = sorted(required - columns)
if missing:
    raise ValueError(f"Colonne mancanti nel dataset: {missing}")

read_cols = [DOC_COL, TEXT_COL]
if REFERENCE_COL in columns:
    read_cols.append(REFERENCE_COL)

paragraphs = dd.read_parquet(
    INPUT,
    columns=read_cols,
    engine="pyarrow",
)

if REFERENCE_COL in paragraphs.columns:
    paragraphs = paragraphs[~paragraphs[REFERENCE_COL].fillna(False)]

paragraphs = paragraphs[[DOC_COL, TEXT_COL]].dropna(subset=[DOC_COL, TEXT_COL])

print("partitions:", paragraphs.npartitions)
paragraphs.head(3)

## 5. Tokenizzazione del testo

Questa cella definisce come trasformare il testo in parole.

La tokenizzazione converte in minuscolo, normalizza alcuni caratteri Unicode, conserva termini scientifici con trattino o slash come `covid-19` e `sars-cov-2`, rimuove stopword inglesi comuni e ignora token troppo corti.

In [ ]:
import re
import unicodedata
from collections import Counter, defaultdict

TOKEN_PATTERN = re.compile(r"[a-z][a-z0-9]+(?:[-/][a-z0-9]+)*")
MIN_TOKEN_LEN = 2

STOPWORDS = set("""
a about above after again against all am an and any are as at be because been
before being below between both but by can could did do does doing down during
each few for from further had has have having he her here hers herself him
himself his how i if in into is it its itself just me more most my myself no
nor not now of off on once only or other our ours ourselves out over own same
she should so some such than that the their theirs them themselves then there
these they this those through to too under until up very was we were what when
where which while who whom why will with you your yours yourself yourselves
also al et fig figure table using use used may one two three however within
without among across per
""".split())

UNICODE_TRANSLATION = str.maketrans({
    "\u2010": "-",
    "\u2011": "-",
    "\u2012": "-",
    "\u2013": "-",
    "\u2014": "-",
    "\u2015": "-",
    "\u2212": "-",
    "\u2018": "'",
    "\u2019": "'",
    "\u201c": '"',
    "\u201d": '"',
})

def normalize_text(text):
    return unicodedata.normalize("NFKC", text).lower().translate(UNICODE_TRANSLATION)

def tokenize(text):
    for token in TOKEN_PATTERN.findall(normalize_text(text)):
        if len(token) >= MIN_TOKEN_LEN and token not in STOPWORDS:
            yield token

## 6. Funzione Map per partizione

Dask passa a questa funzione un piccolo DataFrame pandas per ogni partizione.

La funzione produce righe nella forma:

```text
cord_uid, word, count
```

Questo e' il primo passo MapReduce: i conteggi sono ancora parziali e verranno aggregati nelle celle successive.

In [ ]:
def empty_count_frame():
    return pd.DataFrame({
        DOC_COL: pd.Series(dtype="object"),
        "word": pd.Series(dtype="object"),
        "count": pd.Series(dtype="int64"),
    })

def count_partition(pdf):
    if pdf.empty:
        return empty_count_frame()

    by_doc = defaultdict(Counter)

    for doc_id, text in zip(pdf[DOC_COL], pdf[TEXT_COL]):
        if doc_id is None or not isinstance(text, str) or not text:
            continue
        by_doc[str(doc_id)].update(tokenize(text))

    rows = [
        (doc_id, word, int(count))
        for doc_id, counts in by_doc.items()
        for word, count in counts.items()
    ]

    if not rows:
        return empty_count_frame()

    return pd.DataFrame(rows, columns=[DOC_COL, "word", "count"]).astype({
        DOC_COL: "object",
        "word": "object",
        "count": "int64",
    })

## 7. Smoke test su poche righe

Prima di lanciare Dask su tutto il dataset, questa cella testa la funzione su poche righe caricate in pandas.

Serve a verificare la logica senza fare un calcolo pesante.

In [ ]:
sample = paragraphs.head(200)
test_counts = count_partition(sample)

test_counts.head(20)

## 8. Costruzione lazy del grafo MapReduce

Questa cella prepara il grafo di calcolo, ma non esegue ancora tutto.

Passaggi logici:

1. Map: da paragrafi a conteggi parziali `(cord_uid, word, count)`;
2. Reduce per documento: somma per `(cord_uid, word)`;
3. Reduce globale: somma per `word` sull'intero corpus.

In [ ]:
meta = empty_count_frame()

partial_doc_counts = paragraphs.map_partitions(
    count_partition,
    meta=meta,
)

doc_counts = (
    partial_doc_counts
    .groupby([DOC_COL, "word"])["count"]
    .sum(split_out=SPLIT_OUT)
    .reset_index()
)

global_counts = (
    doc_counts
    .groupby("word")["count"]
    .sum(split_out=SPLIT_OUT)
    .reset_index()
)

global_counts

## 9. Preview piccola su poche partizioni

Questa cella esegue davvero un calcolo, ma solo sulle prime partizioni.

Usala per verificare che il workflow funzioni prima di lanciare il run completo.

In [ ]:
SMOKE_PARTITIONS = min(3, paragraphs.npartitions)

small_paragraphs = paragraphs.partitions[:SMOKE_PARTITIONS]
small_partial = small_paragraphs.map_partitions(
    count_partition,
    meta=empty_count_frame(),
)

small_global = (
    small_partial
    .groupby("word")["count"]
    .sum(split_out=4)
    .reset_index()
)

small_global.nlargest(20, "count").compute()

## 10. Benchmark locale su subset crescenti

Il run completo sull'intero dataset puo richiedere molta memoria in locale. Per questo motivo, in questa fase uso un benchmark controllato su un numero crescente di partizioni del dataset Parquet.

La cella esegue lo stesso workflow MapReduce del word count completo, ma su subset progressivi:

- 3 partizioni;
- 10 partizioni;
- 25 partizioni;
- 50 partizioni.

Per ogni subset vengono misurati:

- numero di partizioni elaborate;
- numero di worker Dask;
- numero di thread per worker;
- valore di `split_out`;
- tempo totale di esecuzione;
- parola piu frequente nel subset;
- conteggio della parola piu frequente.

Gli output vengono salvati in:

```text
Giulia/reports/giulia_word_count/benchmark_local_partitions.csv
Giulia/reports/giulia_word_count/top_words.csv
Giulia/reports/giulia_word_count/summary.json
```

Questo benchmark e' locale e serve a validare il metodo e osservare come cresce il tempo di esecuzione aumentando la quantita di dati. Il run completo distribuito puo essere eseguito successivamente su CloudVeneto, usando lo stesso approccio, se richiesto dalla consegna.


In [ ]:
benchmark_csv = OUT / "benchmark_local_partitions.csv"
top_csv = OUT / "top_words.csv"
summary_json = OUT / "summary.json"

import gc

def worker_sweep():
    import gc
    try:
        import pyarrow as pa
        pa.default_memory_pool().release_unused()
    except Exception:
        pass
    return gc.collect()

benchmark_rows = []
top_words = None
partition_plan = [3, 10, 25, 50]
seen = set()

for requested_parts in partition_plan:
    n_parts = min(requested_parts, paragraphs.npartitions)
    if n_parts in seen:
        continue
    seen.add(n_parts)

    print(f"Benchmark su {n_parts} partizioni...")
    start = time.perf_counter()

    bench_paragraphs = paragraphs.partitions[:n_parts]

    bench_partial = bench_paragraphs.map_partitions(
        count_partition,
        meta=empty_count_frame(),
    )

    bench_doc_counts = (
        bench_partial
        .groupby([DOC_COL, "word"])["count"]
        .sum(split_out=2)
        .reset_index()
    )

    bench_global = (
        bench_doc_counts
        .groupby("word")["count"]
        .sum(split_out=2)
        .reset_index()
    )

    current_top = bench_global.nlargest(TOP_N, "count").compute()
    current_top = current_top.sort_values("count", ascending=False).reset_index(drop=True)

    elapsed = time.perf_counter() - start
    top_words = current_top

    benchmark_rows.append({
        "n_partitions": n_parts,
        "n_workers": len(client.scheduler_info()["workers"]),
        "threads_per_worker": 1,
        "split_out": 2,
        "top_n": TOP_N,
        "elapsed_seconds": round(elapsed, 2),
        "top_word": str(current_top.iloc[0]["word"]) if len(current_top) else None,
        "top_word_count": int(current_top.iloc[0]["count"]) if len(current_top) else 0,
    })

    del bench_paragraphs, bench_partial, bench_doc_counts, bench_global, current_top
    gc.collect()

    try:
        client.run(worker_sweep)
    except Exception:
        pass

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(benchmark_csv, index=False)

if top_words is None:
    raise RuntimeError("Benchmark non eseguito: controlla che il dataset abbia partizioni disponibili.")

top_words.to_csv(top_csv, index=False)

summary = {
    "input": str(INPUT),
    "output": str(OUT),
    "mode": "local_subset_benchmark",
    "benchmark_csv": str(benchmark_csv),
    "top_words_csv": str(top_csv),
    "top_n": TOP_N,
    "partition_plan": sorted(seen),
    "note": "Benchmark locale su subset crescenti. Il run completo distribuito va eseguito su CloudVeneto/cluster se richiesto dalla consegna.",
}

summary_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")

display(benchmark_df)
top_words

## 11. Grafico delle top parole del benchmark locale

Questa cella crea un grafico delle parole piu frequenti ottenute dall'ultimo subset elaborato nella cella precedente.

Nel notebook locale, `top_words` non rappresenta necessariamente l'intero corpus CORD-19, ma il subset piu grande usato nel benchmark locale, ad esempio 50 partizioni.

Il grafico viene salvato in:

```text
Giulia/reports/giulia_word_count/top_words.png
```

Il grafico e' utile per controllare qualitativamente il risultato del word count e per visualizzare le parole piu frequenti del subset analizzato.


In [ ]:
import matplotlib.pyplot as plt

plot_path = OUT / "top_words.png"

plot_df = top_words.sort_values("count", ascending=True)
height = max(5, 0.34 * len(plot_df))

fig, ax = plt.subplots(figsize=(11, height))
ax.barh(plot_df["word"], plot_df["count"], color="#2f6f73")
ax.set_title(f"Top {len(top_words)} words in local benchmark subset")
ax.set_xlabel("occurrences")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(plot_path, dpi=150)
plt.show()

print("saved:", plot_path)

## 12. Chiusura del cluster Dask

Quando hai finito, chiudi client e cluster per liberare memoria sul Mac.

In [ ]:
client.close()
cluster.close()
print("Dask chiuso")